# 260428 LangGraph 상태 그래프 구현

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w8_langgraph/llm_260428_state_graph.ipynb)

In [ ]:
!pip install -q langchain langchain-community langchain-core langchain-openai openai

In [ ]:
# --- Colab 전용 ---
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

from openai import OpenAI
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

client = OpenAI()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

## 강의 메모: LangGraph 상태 그래프 핵심 (State / Node / Edge)

- **왜 LangGraph인가**: LCEL 체인은 `prompt | retriever | llm | parser`처럼 한 방향 직선. 하지만 에이전트는 환불/기술 분기, 검색 결과 부족 시 루프, 여러 LLM 병렬 후 머지 등이 필요 → 직선 체인은 `if/else` 덕지덕지로 너덜너덜해짐. 그래프로 분기·루프·병렬을 자연스럽게.
- **지하철 노선도 비유**: 삼성역(노드) → 강남역(노드)을 잇는 선(엣지). 단 LangGraph에서 **"노드는 함수"** — 처음 배울 때 가장 많이 헷갈리는 포인트.
- **3대 구성요소**: ① **State**(그래프 전체 공유 dict, TypedDict로 정의) ② **Node**(state 받아 변경된 키만 리턴하는 함수) ③ **Edge**(노드 연결선). 모든 그래프는 `START`에서 출발 `END`로 끝.
- **State 설계가 70~80%**: 그래프 품질 대부분이 State 스키마에서 결정됨. `TypedDict`는 필수는 아니지만 `age: int`에 `"35"` 들어오면 즉시 에러 → 운영 중 발견 전에 잡힘.
- **노드 함수 규칙 — 변경된 키만 리턴**: `return {'count': state['count']+1}`처럼 업데이트된 키만. 전체 state 리턴은 동작은 하지만 **그래프가 커지면 LangGraph 내부 머지 로직과 충돌해 에러**. 나머지 키는 자동 merge됨.
- **컴파일 1회, invoke N회**: `app = builder.compile()`은 한 번. 이후 같은 `app`을 여러 번 invoke 가능.
- **노드 안은 자유**: LangGraph는 들어오는 state, 나가는 dict만 본다. LLM 호출·retriever·외부 API 다 노드 함수 안에 자유롭게.
- **분기는 `add_conditional_edges` + 라우터**: 라우터는 **노드가 아니라 문자열을 리턴하는 함수**. 시작점은 항상 노드여야 하므로 관례적으로 빈 dict만 리턴하는 `start_n` 같은 노드를 두고 거기서 분기.
- **머지 패턴**: 분기된 여러 브랜치를 동일한 `merge` 노드로 모아 후처리 후 `END`로. 분기 후 공통 후처리의 정석.

## 강의 메모: 실무 팁

- **라우터 리턴값 ↔ conditional_edges 딕셔너리 키 100% 일치**: 라우터가 `'pass'`,`'fail'` 리턴하는데 매핑에 `'fail'` 누락 시 그래프 사망. 강사도 실제로 이전 코드의 `'even_handler'`가 남아 분기가 어긋나 디버깅함. **리턴값 집합 == 딕셔너리 키 집합** 체크 습관화.
- **TypedDict 문법 함정**: `length = int`(대입) vs `length: int`(타입 어노테이션) — 강의 중 실제 발생한 오타. 전자는 단순 변수 대입이라 State 키로 잡히지 않고 결과에서 사라짐. **콜론 vs 등호** 꼭 확인.
- **빈 노드(start_n) 두는 이유**: `add_conditional_edges`의 시작점은 반드시 노드여야 함. `START`→라우터 직결도 가능은 하나, 명시적으로 시작 노드를 따로 두는 게 관례 → 가독성 + 전처리 추가 여지.
- **디버깅용 log 키 패턴**: State에 `log: str` 키 하나 두고 각 노드에서 `log + 'clean,'`로 누적 → 어떤 노드가 어떤 순서로 실행됐는지 즉시 확인. `stream`/`debug` 모드 쓰기 전 간단 진단으로 유용.
- **invoke 입력은 State 키 전부 채우기**: `app.invoke({'text':'hi','upper':'','length':0})`처럼 빈 값이라도 다 넣어주는 게 안전. 나중에 업데이트될 키도 초기값 명시.
- **분기+머지 = 에이전트 기본기**: `if/else`로도 가능하지만 그래프로 짜야 병렬 호출, 재실행, 체크포인트로 자연스럽게 확장됨.